In [1]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
botname = 'ds6600'
version = '0.0'
purpose = 'Class demo'
email = 'jkropko@virginia.edu'
useragent = f'{botname}/{version} ({purpose}; {email}) python-requests/{requests.__version__}'
headers = {'User-Agent': useragent}

In [3]:
house_committees_url = 'https://clerk.house.gov/committees/'

r = requests.get(house_committees_url, headers=headers)
r

<Response [200]>

In [6]:
mysoup = BeautifulSoup(r.text, 'html.parser')
#mysoup

Anatomy of HTML

```
<tag attribute="value"> navigable string </tag> 

```

In [10]:
mylist = mysoup.find_all('a', attrs = {'class': 'library-committeePanel-subItems'})

In [17]:
{'url': 'https://clerk.house.gov' + mylist[0]['href'],
 'committee_name': mylist[0].text.strip()}

{'url': 'https://clerk.house.gov/committees/AG22',
 'committee_name': 'Commodity Markets, Digital Assets, and Rural Development'}

In [19]:
committee_df = pd.DataFrame([{'url': 'https://clerk.house.gov' + x['href'],
 'committee_name': x.text.strip()} for x in mylist])
committee_df

,url,committee_name
0,https://clerk.house.gov/committees/AG22,"Commodity Markets, Digital Assets, and Rural D..."
1,https://clerk.house.gov/committees/AG14,"Conservation, Research, and Biotechnology"
2,https://clerk.house.gov/committees/AG15,Forestry and Horticulture
3,https://clerk.house.gov/committees/AG16,"General Farm Commodities, Risk Management, and..."
4,https://clerk.house.gov/committees/AG29,"Livestock, Dairy, and Poultry"
...,...,...
105,https://clerk.house.gov/committees/IG04,Defense Intelligence and Overhead Architecture
106,https://clerk.house.gov/committees/IG06,National Intelligence Enterprise
107,https://clerk.house.gov/committees/IG02,National Security Agency and Cyber
108,https://clerk.house.gov/committees/IG11,Open Source Intelligence


In [22]:
url = committee_df['url'][0]

In [23]:
r = requests.get(url, headers=headers)
r

<Response [200]>

In [25]:
mysoup = BeautifulSoup(r.text, 'html.parser')

In [29]:
mylist = mysoup.find_all('a', attrs = {'title':'View Profile'})

bioguideIds = [x['href'].replace('/Members/', '') for x in mylist]
pd.DataFrame({'committee': committee_df['committee_name'][0],
              'bioguideId': bioguideIds})

,committee,bioguideId
0,"Commodity Markets, Digital Assets, and Rural D...",J000301
1,"Commodity Markets, Digital Assets, and Rural D...",R000612
2,"Commodity Markets, Digital Assets, and Rural D...",L000491
3,"Commodity Markets, Digital Assets, and Rural D...",S001189
4,"Commodity Markets, Digital Assets, and Rural D...",R000603
5,"Commodity Markets, Digital Assets, and Rural D...",M000871
6,"Commodity Markets, Digital Assets, and Rural D...",C001039
7,"Commodity Markets, Digital Assets, and Rural D...",F000475
8,"Commodity Markets, Digital Assets, and Rural D...",N000193
9,"Commodity Markets, Digital Assets, and Rural D...",B001327


In [31]:
def congress_spider(url, committee):
    print(committee)
    r = requests.get(url, headers=headers)
    mysoup = BeautifulSoup(r.text, 'html.parser')
    mylist = mysoup.find_all('a', attrs = {'title':'View Profile'})
    bioguideIds = [x['href'].replace('/Members/', '') for x in mylist]
    return {'committee': committee,
              'bioguideId': bioguideIds}

In [33]:
full_committee_list = [congress_spider(u,c) 
                       for u, c in zip(committee_df['url'], committee_df['committee_name'])]

Commodity Markets, Digital Assets, and Rural Development
Conservation, Research, and Biotechnology
Forestry and Horticulture
General Farm Commodities, Risk Management, and Credit
Livestock, Dairy, and Poultry
Nutrition and Foreign Agriculture
Agriculture, Rural Development, Food and Drug Administration, and Related Agencies
Commerce, Justice, Science, and Related Agencies
Defense
Energy and Water Development and Related Agencies
Financial Services and General Government
Homeland Security
Interior, Environment, and Related Agencies
Labor, Health and Human Services, Education, and Related Agencies
Legislative Branch
Military Construction, Veterans Affairs, and Related Agencies
National Security, Department of State, and Related Programs
Transportation, Housing and Urban Development, and Related Agencies
Cyber, Information Technologies, and  Innovation
Intelligence and Special Operations
Military Personnel
Readiness
Seapower and Projection Forces
Strategic Forces
Tactical Air and Land For

In [ ]:
list_of_dfs = [pd.DataFrame(x) for x in full_committee_list]
#list_of_dfs[1]
full_df = pd.concat(list_of_dfs)
full_df

,committee,bioguideId
0,"Conservation, Research, and Biotechnology",L000491
1,"Conservation, Research, and Biotechnology",B001307
2,"Conservation, Research, and Biotechnology",B001298
3,"Conservation, Research, and Biotechnology",F000446
4,"Conservation, Research, and Biotechnology",M001211
5,"Conservation, Research, and Biotechnology",R000612
6,"Conservation, Research, and Biotechnology",J000304
7,"Conservation, Research, and Biotechnology",N000189
8,"Conservation, Research, and Biotechnology",B001327
9,"Conservation, Research, and Biotechnology",M001233
